# Samenstellen van een DNA sequentie

Omdat DNA-sequencing technieken enkel relatief korte stukken DNA kunnen lezen (100-1000 nucleotiden), is het nodig om deze korte stukjes DNA op een correcte manier te kunnen verbinden om zo het volledige genoom te kunnen reconstrueren.

In deze notebook vertrek je van een lijst van deelsequenties van het genoom van het MS2 virus. Het doel is om deze sequenties terug samen te voegen tot het volledige genoom.

Het MS2 virus is een virus dat bacteriën aanvalt (een bacteriofaag). Het is interessant voor onderzoekers omdat het een zeer kort genoom heeft dat bestaat uit maar 3569 nucleotiden.

Eerst importeren we de nodige bibliotheken die we nodig zullen hebben in de notebook.

In [1]:
import random
from collections import defaultdict
from scripts import helpers

## De deelsequenties lezen

De deelsequenties kan je vinden in het bestand *ms2_phage_genome_shotgun_30mers.txt* in de map *sequenties*. Op elke lijn van het bestand staat een willekeurig stukje uit het DNA van MS2. Merk op dat deze stukjes vaak overlappen. Ons doel is om deze overlap te vinden en op basis daarvan het volledige genoom te reconstrueren.

In onderstaande codecel worden de stukjes DNA ingelezen in een lijst.

In [ ]:
# Maak een lijst om alle DNA sequenties in op te slaan
sub_sequenties = []

# Open het bestand met de sequenties en voeg ze toe aan de lijst
with open('sequenties/ms2_phage_genome_shotgun_30mers.txt', 'r') as f:
    for line in f:
        sub_sequenties.append(line.strip())
        
# Druk het aantal subsequenties af
print(len(sub_sequenties))

Zoals je kan zien zitten er in het bestand 10000 sequenties die deel zijn van het MS2 genoom. 

We kunnen ook zoeken naar de kortste en de langste sequentie in het bestand.

In [ ]:
# Zoek de lengte van de kortste en langste sequentie
min_len = len(sub_sequenties[0])
max_len = len(sub_sequenties[0])
for seq in sub_sequenties:
    if len(seq) < min_len:
        min_len = len(seq)
    if len(seq) > max_len:
        max_len = len(seq)
        
print(f"De korste sequentie is {min_len} nucleotiden lang.")
print(f"De langste sequentie is {max_len} nucleotiden lang.")

Het bestand bestaat dus uit sequenties van 50 tot 100 nucleotiden lang. We kunnen 5 willekeurige sequenties bekijken.

In [ ]:
# Druk 5 willekeurige sequenties af
for i in range(5):
    print(random.choice(sub_sequenties))
    

## Een De Bruijn graaf opstellen

Met onze lijst van sequenties kunnen we een De Bruijn graaf opstellen. 

De De Bruijn graaf maakt de veronderstelling dat alle sequenties die je in de boom wil opslaan even veel nucleotiden bevatten. Onze lijst bevat echter sequenties van verschillende lengtes tussen 50 en 100 nucleotiden. Om dit probleem op te lossen kiezen we een waarde *k* die korter is dan de kortste sequentie in onze lijst (50 nucleotiden). We delen ons stuk DNA dan op in overlappende stukken van *k* nucleotiden lang (k-meren). Op onderstaande figuur zie je hoe dat gaat voor k=3. In de figuur wordt de sequentie GAGCTTTTAG opgesplitst in 8 3-meren.

![](img/splitting_dna_into_overlapping_sequences.svg)

De volgende functie zal een lijst van sequenties toevoegen aan een De Bruijn graaf. De functie zal elke sequentie opdelen in k-meren. Voor elk k-meer wordt de prefix van lengte k-1 en de suffix van lengte k-1 toegevoegd aan de graaf (indien die nog niet in de graaf zit). Tussen deze knopen komt een boog.

In [5]:
# Deze functie bouwt een de Bruijn graaf van de sequenties
def stel_de_bruijn_graaf_op(sequenties, k):
    # We stellen de graaf voor als een dictionary met als keys de prefixen en als values de suffixen
    graaf = defaultdict(list)
    # Overloop alle sequenties
    for sequentie in sequenties:
        # Voeg de sequentie enkel toe als die langer is dan de k-meer lengte.
        if len(sequentie) >= k:
            # Splits de sequentie op in k-meren.
            for i in range(len(sequentie) - k + 1):
                kmer = sequentie[i:i+k]
                # Neem alle letters, behalve de laatste, als waarde voor een knoop in de graaf.
                prefix = kmer[:-1]
                # Neem alle letters, behalve de eerste, als waarde voor een knoop in de graaf.
                suffix = kmer[1:]
                # Voeg een boog toe tussen de prefix en de suffix als die nog niet bestaat.
                if suffix not in graaf[prefix]: 
                    graaf[prefix].append(suffix)
    return graaf

**Opdracht:** Gebruik de functie *stel_de_bruijn_graaf_op* om een De Bruijn graaf op te stellen van de ingeladen *sub_sequenties*. Gebruik k-meren van lengte 20.

In [6]:
de_bruijn_graaf = # Vul hier de functieoproep in

## Op zoek naar het euleriaans pad

Een euleriaans pad in deze graaf komt overeen met de reconstructie van het origineel genoom. Voor we een euleriaans pad kunnen vinden moeten we controleren of er zo'n pad te vinden is. Weet jij nog aan welke voorwaarden een graaf moet voldoen om een euleriaans pad te hebben?

In [ ]:
# We tellen het aantal knopen in de graaf met een oneven graad
graden = defaultdict(int)

# Bereken voor elke knoop hoeveel bogen er toekomen en vertrekken
for knoop in de_bruijn_graaf:
    for buur in de_bruijn_graaf[knoop]:
        graden[knoop] += 1
        graden[buur] += 1
        
# Tel het aantal knopen met een oneven graad
knopen_met_oneven_graad = 0
for knoop in graden.keys():
    if graden[knoop] % 2 != 0:
        knopen_met_oneven_graad += 1
        
        
# Druk het aantal knopen met een oneven graad af
print(knopen_met_oneven_graad)


**Opdracht:** Kunnen we in deze graaf een euleriaans pad terugvinden? Zoja, waarom?

Wanneer je hebt bevestigd dat er een euleriaans pad bestaat in de graaf, kunnen we een algoritme zoeken om dat pad te vinden. Je kent het Algoritme van Fleury al. Dat algoritme kost echter veel rekenkracht. Daarom gebruiken we hier achter de schermen een ander algoritme, het algoritme van Hierholzer. Wil je weten hoe dat algoritme werkt? Dan kan je een kijkje nemen in het *helpers.py* bestand in de *scripts* map.

Voer onderstaande codecel uit om het eulerpad in de De Bruijn graaf te zoeken.

In [8]:
eulerpad = helpers.zoek_eulerpad(de_bruijn_graaf)

**Opdracht:** Druk de knopen op het pad af.

In [ ]:
# Schrijf hier de code om het Eulerpad af te drukken.

**Opdracht:** Druk het aantal knopen op het pad af.

In [ ]:
# Schrijf hier de code om het aantal elementen van het Eulerpad af te drukken.

Bekijk het aantal knopen op het pad. Dit komt niet overeen met het aantal nucleotiden in het oorspronkelijke genoom. Dat bevat 3569 nucleotiden. 

**Opdracht:** Kan je het verschil verklaren?

## Het genoom reconstrueren

Het pad bevat prefixen en suffixen van de k-meren die we uit de stukjes van het genoom hebben geknipt. In die prefixen en suffixen zit veel overlap. We kunnen de knopen dus niet eenvoudig achter elkaar plakken.

**Opdracht:** Vul de functie *reconstrueer_genoom* aan zodat die de overlappende informatie uit verschillende knopen verwijdert en de overige informatie in de knopen samenvoegt tot één tekst (het volledige genoom).

In [15]:
# Step 4: Reconstruct the genome from the Eulerian path
def reconstrueer_genoom(pad):
    # vul hier de code in om het genoom te reconstrueren
    return genoom

Je kan nu je functie oproepen en het gereconstrueerde genoom afdrukken.

In [ ]:
gereconstrueerd_genoom = reconstrueer_genoom(eulerpad)
print(gereconstrueerd_genoom)
print(len(gereconstrueerd_genoom))

## De reconstructie controleren

Wanneer je DNA-sequencing doet op een nieuw genoom is deze stap niet mogelijk omdat je de correcte informatie niet hebt. Wij kunnen het gekende genoom van het MS2 virus echter wel gebruiken om te controleren of onze reconstructie correct is. De sequentie van het genoom komt uit de [NIH bibliotheek](https://www.ncbi.nlm.nih.gov/nuccore/NC_001417.2?report=fasta).

Het bestaande genoom kunnen we inladen uit het bestand *ms2_phage_genome.txt* in de map *sequenties*.

In [19]:
# Lees het originele genoom in uit het bestand sequenties/ms2_phage_genome.txt
with open('sequenties/ms2_phage_genome.txt', 'r') as f:
    origineel_genoom = f.readline().strip()

**Opdracht:** Druk het gereconstrueerde genoom en het originele genoom onder elkaar af.

In [ ]:
# Schrijf hier de code om de genomen onder elkaar af te drukken.

Je kan nu ofwel voor alle 3569 nucleotiden controleren of ze overeenkomen of je kan Python gebruiken om dat te checken.

In [ ]:
if gereconstrueerd_genoom == origineel_genoom:
    print("Het gereconstrueerde genoom is correct.")
else:
    print("Het gereconstrueerde genoom bevat fouten.")

# Partners

Dit materiaal werd ontwikkeld door Dwengo vzw en kwam tot stand met steun van VLAIO. Vind al het materiaal van ons wAIsda project op [dwengo.org/waisda](dwengo.org/waisda)

!["VLAIO logo"](img/vlaio.png)

!["Dwengo logo"](img/dwengo-groen-zwart.png)